# Algorithms 35-36: Artificial Neural Networks**Learning Objectives:**1. Design a Multi-Layer Perceptron (MLP)2. Implement Forward Propagation using Matrix multiplications3. Implement Backpropagation to calculate gradients across hidden layers4. Train a neural network to solve the non-linear XOR problem**Prerequisites:** Algorithms 27-28**Estimated time:** 120 minutes**Textbook:** *Justin Skycak — Intro to Algorithms & Machine Learning* Chapters 35-36

In [ ]:
# Google Colab Setup!pip install -q ipywidgets jupyterquiz ipytest plotly sympy networkx pandas numpy matplotlib tqdmimport json, os, sysfrom pathlib import Pathif Path('/content').exists():    repo_root = Path('/content/upskill')else:    repo_root = Path.cwd()if str(repo_root) not in sys.path:    sys.path.insert(0, str(repo_root))try:    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )except ModuleNotFoundError:    Path('learning_tools.py').write_text('"""Colab-first learning helpers for the interactive math curriculum.\n\nThe functions in this module are intentionally lightweight: they work in a\nplain notebook, in Google Colab, and in local Jupyter without requiring a\nbackend service.\n"""\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, asdict, field\nfrom datetime import datetime, timedelta, timezone\nimport json\nimport math\nimport os\nimport re\nfrom pathlib import Path\nfrom typing import Any, Callable, Iterable\n\n\nSCHEMA_VERSION = 1\nDEFAULT_PROGRESS_FILENAME = "upskill_math_progress.json"\nDRIVE_PROGRESS_DIR = "MyDrive/upskill"\n\n\ndef _utcnow() -> datetime:\n    return datetime.now(timezone.utc)\n\n\ndef _iso(dt: datetime) -> str:\n    return dt.astimezone(timezone.utc).replace(microsecond=0).isoformat()\n\n\ndef _parse_iso(value: str | None) -> datetime | None:\n    if not value:\n        return None\n    try:\n        return datetime.fromisoformat(value.replace("Z", "+00:00"))\n    except ValueError:\n        return None\n\n\n@dataclass\nclass Skill:\n    """A small curriculum skill descriptor."""\n\n    id: str\n    title: str\n    notebook: str\n    level: int\n    prerequisites: list[str] = field(default_factory=list)\n    tags: list[str] = field(default_factory=list)\n\n    def to_dict(self) -> dict[str, Any]:\n        return asdict(self)\n\n\ndef setup_colab(\n    progress_filename: str = DEFAULT_PROGRESS_FILENAME,\n    mount_drive: bool = True,\n    verbose: bool = True,\n) -> Path:\n    """Prepare a notebook session and return the progress-file path.\n\n    In Colab, this mounts Google Drive when available. Outside Colab, it falls\n    back to the current working directory.\n    """\n\n    progress_path = Path(progress_filename)\n    try:\n        import google.colab  # type: ignore  # noqa: F401\n        in_colab = True\n    except Exception:\n        in_colab = False\n\n    if in_colab and mount_drive:\n        try:\n            from google.colab import drive  # type: ignore\n\n            drive.mount("/content/drive")\n            drive_dir = Path("/content/drive") / DRIVE_PROGRESS_DIR\n            drive_dir.mkdir(parents=True, exist_ok=True)\n            progress_path = drive_dir / progress_filename\n        except Exception as exc:\n            if verbose:\n                print(f"Drive mount skipped; using local progress file. ({exc})")\n\n    if verbose:\n        print(f"Learning environment ready. Progress: {progress_path}")\n    return progress_path\n\n\nclass ProgressStore:\n    """JSON-backed spaced-repetition progress store."""\n\n    def __init__(self, path: str | Path | None = None, learner_id: str = "local"):\n        self.path = Path(path or DEFAULT_PROGRESS_FILENAME)\n        self.learner_id = learner_id\n        self.data = self._load()\n\n    def _default_data(self) -> dict[str, Any]:\n        return {\n            "schema_version": SCHEMA_VERSION,\n            "learner_id": self.learner_id,\n            "skills": {},\n        }\n\n    def _load(self) -> dict[str, Any]:\n        if not self.path.exists():\n            return self._default_data()\n        try:\n            data = json.loads(self.path.read_text(encoding="utf-8"))\n        except Exception:\n            return self._default_data()\n        if data.get("schema_version") != SCHEMA_VERSION:\n            data["schema_version"] = SCHEMA_VERSION\n        data.setdefault("learner_id", self.learner_id)\n        data.setdefault("skills", {})\n        return data\n\n    def save(self) -> None:\n        self.path.parent.mkdir(parents=True, exist_ok=True)\n        self.path.write_text(json.dumps(self.data, indent=2), encoding="utf-8")\n\n    def get(self, skill_id: str) -> dict[str, Any]:\n        return self.data.setdefault("skills", {}).setdefault(skill_id, {\n            "attempts": 0,\n            "correct": 0,\n            "confidence": 0,\n            "mastery": 0.0,\n            "last_seen": None,\n            "next_review": None,\n        })\n\n    def record_attempt(\n        self,\n        skill_id: str,\n        correct: bool,\n        confidence: int = 3,\n        item_type: str = "practice",\n    ) -> dict[str, Any]:\n        confidence = max(1, min(5, int(confidence)))\n        row = self.get(skill_id)\n        attempts = int(row.get("attempts", 0)) + 1\n        correct_count = int(row.get("correct", 0)) + int(bool(correct))\n        accuracy = correct_count / attempts\n        confidence_factor = confidence / 5\n        mastery = round((0.75 * accuracy) + (0.25 * confidence_factor), 3)\n\n        row.update({\n            "attempts": attempts,\n            "correct": correct_count,\n            "confidence": confidence,\n            "mastery": mastery,\n            "last_seen": _iso(_utcnow()),\n            "last_item_type": item_type,\n            "next_review": _iso(_utcnow() + review_interval(correct, confidence, attempts)),\n        })\n        self.save()\n        return row\n\n    def review_queue(\n        self,\n        skills: Iterable[Skill | dict[str, Any]] | None = None,\n        limit: int = 8,\n        now: datetime | None = None,\n    ) -> list[dict[str, Any]]:\n        now = now or _utcnow()\n        known_titles = {}\n        if skills:\n            for skill in skills:\n                item = skill.to_dict() if isinstance(skill, Skill) else skill\n                known_titles[item["id"]] = item\n\n        due: list[dict[str, Any]] = []\n        for skill_id, row in self.data.get("skills", {}).items():\n            next_review = _parse_iso(row.get("next_review"))\n            if next_review is None or next_review <= now:\n                due.append({\n                    "id": skill_id,\n                    "title": known_titles.get(skill_id, {}).get("title", skill_id),\n                    "mastery": row.get("mastery", 0.0),\n                    "next_review": row.get("next_review"),\n                    "attempts": row.get("attempts", 0),\n                })\n        due.sort(key=lambda item: (item["mastery"], item["next_review"] or ""))\n        return due[:limit]\n\n    def weak_skills(self, limit: int = 8) -> list[tuple[str, dict[str, Any]]]:\n        rows = list(self.data.get("skills", {}).items())\n        rows.sort(key=lambda kv: (kv[1].get("mastery", 0.0), -kv[1].get("attempts", 0)))\n        return rows[:limit]\n\n\ndef review_interval(correct: bool, confidence: int, attempts: int = 1) -> timedelta:\n    """Return the next review interval from correctness and confidence."""\n\n    if not correct:\n        return timedelta(hours=6) if attempts <= 1 else timedelta(days=1)\n    if confidence <= 2:\n        return timedelta(days=1)\n    if confidence == 3:\n        return timedelta(days=3)\n    high_confidence_days = [7, 14, 30, 60]\n    return timedelta(days=high_confidence_days[min(max(attempts - 1, 0), len(high_confidence_days) - 1)])\n\n\ndef record_attempt(\n    skill_id: str,\n    correct: bool,\n    confidence: int = 3,\n    item_type: str = "practice",\n    store: ProgressStore | None = None,\n) -> dict[str, Any]:\n    store = store or ProgressStore()\n    return store.record_attempt(skill_id, correct, confidence, item_type)\n\n\ndef check(\n    name: str,\n    actual: Any,\n    expected: Any,\n    tolerance: float | None = None,\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Friendly equality/numeric check with optional progress recording."""\n\n    if tolerance is not None and isinstance(actual, (int, float)) and isinstance(expected, (int, float)):\n        ok = math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance)\n    else:\n        ok = actual == expected\n\n    if ok:\n        print(f"PASS: {name}")\n    else:\n        print(f"TRY AGAIN: {name}")\n        print(f"  expected: {expected!r}")\n        print(f"  actual:   {actual!r}")\n\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "check", store)\n    return ok\n\n\ndef code_check(\n    name: str,\n    fn: Callable[..., Any],\n    cases: Iterable[tuple[tuple[Any, ...], Any] | tuple[tuple[Any, ...], dict[str, Any], Any]],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    """Run multiple function cases and report the first failure."""\n\n    all_ok = True\n    for index, case in enumerate(cases, 1):\n        if len(case) == 2:\n            args, expected = case  # type: ignore[misc]\n            kwargs = {}\n        else:\n            args, kwargs, expected = case  # type: ignore[misc]\n        try:\n            actual = fn(*args, **kwargs)\n            ok = actual == expected\n        except Exception as exc:\n            actual = f"{type(exc).__name__}: {exc}"\n            ok = False\n        if not ok:\n            all_ok = False\n            print(f"TRY AGAIN: {name} case {index}")\n            print(f"  args:     {args}")\n            print(f"  expected: {expected!r}")\n            print(f"  actual:   {actual!r}")\n            break\n    if all_ok:\n        print(f"PASS: {name} ({index} cases)")\n    if skill_id:\n        record_attempt(skill_id, all_ok, confidence, "code_check", store)\n    return all_ok\n\n\ndef normalize_answer(value: Any) -> str:\n    text = str(value).strip().lower()\n    text = re.sub(r"\\s+", " ", text)\n    text = re.sub(r"[^a-z0-9.+/=\\- ]", "", text)\n    text = re.sub(r"\\s*=\\s*", "=", text)\n    return text\n\n\ndef short_answer_check(\n    name: str,\n    answer: Any,\n    accepted: Iterable[Any],\n    skill_id: str | None = None,\n    confidence: int = 3,\n    store: ProgressStore | None = None,\n) -> bool:\n    normalized = normalize_answer(answer)\n    accepted_norm = {normalize_answer(item) for item in accepted}\n    ok = normalized in accepted_norm\n    print(f"{\'PASS\' if ok else \'TRY AGAIN\'}: {name}")\n    if not ok:\n        print("  Re-read the prompt and try to retrieve the answer before opening a hint.")\n    if skill_id:\n        record_attempt(skill_id, ok, confidence, "short_answer", store)\n    return ok\n\n\ndef hint_box(title: str, hints: list[str], unlock: bool = False) -> None:\n    """Display staged hints. Full hints require unlock=True."""\n\n    print(title)\n    if not hints:\n        print("No hints for this item.")\n        return\n    if not unlock:\n        print(f"Hint 1: {hints[0]}")\n        print("Run again with unlock=True after a real attempt to see more.")\n        return\n    for index, hint in enumerate(hints, 1):\n        print(f"Hint {index}: {hint}")\n\n\ndef mastery_dashboard(\n    store: ProgressStore | None = None,\n    skills: Iterable[Skill | dict[str, Any]] | None = None,\n    next_notebook: str | None = None,\n) -> None:\n    store = store or ProgressStore()\n    due = store.review_queue(skills=skills)\n    weak = store.weak_skills()\n    total = len(store.data.get("skills", {}))\n    mastered = sum(1 for row in store.data.get("skills", {}).values() if row.get("mastery", 0) >= 0.8)\n\n    print("Mastery Dashboard")\n    print("=================")\n    print(f"Skills attempted: {total}")\n    print(f"Skills at mastery >= 0.80: {mastered}")\n    if due:\n        print("\\nDue for review:")\n        for item in due:\n            print(f"- {item[\'title\']} (mastery {item[\'mastery\']})")\n    else:\n        print("\\nNo due reviews yet.")\n    if weak:\n        print("\\nWeakest skills:")\n        for skill_id, row in weak:\n            print(f"- {skill_id}: mastery {row.get(\'mastery\', 0)} after {row.get(\'attempts\', 0)} attempts")\n    if next_notebook:\n        print(f"\\nRecommended next notebook: {next_notebook}")\n', encoding='utf-8')    from learning_tools import (        ProgressStore, Skill, setup_colab, check, code_check,        short_answer_check, hint_box, mastery_dashboard    )progress_path = setup_colab()store = ProgressStore(progress_path)print('Ready for retrieval practice.')

In [ ]:
NOTEBOOK = {    "id": "35-36_neural_networks",    "level": 6,    "title": "Algorithms 35-36: Artificial Neural Networks",    "prerequisites": [        "Algorithms 27-28"    ],    "skills_taught": [        "lesson.algorithms_35_36_neural_networks"    ],    "skills_practiced": [        "py.arithmetic.expressions"    ],    "next_notebook": "37-39_minimax_game_trees.ipynb"}SKILLS = [    {        "id": "lesson.algorithms_35_36_neural_networks",        "title": "Lesson Algorithms 35 36 Neural Networks",        "notebook": "35-36_neural_networks.ipynb",        "level": 6    },    {        "id": "py.arithmetic.expressions",        "title": "Py Arithmetic Expressions",        "notebook": "35-36_neural_networks.ipynb",        "level": 6    }]

## Before You Learn: Retrieval GateClose notes for a moment. Try to pull prerequisite ideas from memory before continuing. Getting stuck is useful data for the spaced-review system.

In [ ]:
due = store.review_queue(skills=SKILLS, limit=5)if due:    print('Due review items:')    for item in due:        print(f"- {item['title']} (mastery {item['mastery']})")else:    print('No due reviews yet. Use the recall prompt below to warm up.')

**Recall prompt.** Before reading, name one key idea or procedure you remember from: Algorithms 27-28.

In [ ]:
recall_answer = ''  # write a one-sentence memory before continuingretrieved = len(recall_answer.strip()) >= 12record = store.record_attempt('lesson.algorithms_35_36_neural_networks', retrieved, confidence=3, item_type='prerequisite_recall')print('Recorded retrieval attempt.' if retrieved else 'Write a real memory first, then rerun this cell.')record

## Learning LoopUse the existing lesson cells below as the micro-lesson and practice surface. For each exercise: predict first, attempt from memory, run checks, then open explanations only after the attempt.

In [ ]:
# Google Colab Setupimport mathimport randomimport matplotlib.pyplot as pltprint('Ready ✅')

---## Phase −1 — Theoretical Foundation### From Logistic Regression to Neural Networks> 📖 *From Algorithms:* A single logistic regression model is essentially a single "neuron". It takes inputs, computes a weighted sum, and passes it through a non-linear activation function.What if we hook up multiple neurons together? - The **Input Layer** receives raw data $(x_1, x_2)$.- The **Hidden Layer** consists of $N$ neurons. Each neuron acts as a logistic regression on the inputs, detecting different hidden "features".- The **Output Layer** is a final neuron that acts as a logistic regression on the *outputs of the hidden layer*.By chaining linear combinations and non-linear activations, a Neural Network can approximate literally *any* mathematical function (Universal Approximation Theorem).### BackpropagationTo train this network via Gradient Descent, we need to know the error gradient for every single weight. For the Output Layer weights, we just use the chain rule exactly like Phase 1 of the Logistic Regression notebook.For the Hidden Layer weights, we use the chain rule again, propagating the error *backwards* from the output layer through the hidden layer's activation function. This algorithmic application of the Chain Rule is called **Backpropagation**.

### Comprehension Check ✅Why is the non-linear activation function (like Logistic $\sigma$, or ReLU) strictly required? What would happen if the neurons just did $z = a_1 x_1 + a_2 x_2$ and passed $z$ straight to the next layer?<details><summary>💡 Explanation after a retrieval attempt</summary>A linear combination of a linear combination is mathematically just a single linear combination. Without non-linear activation functions, a 100-layer neural network collapses into a simple $y = mx+b$ linear regression, and completely loses its ability to learn complex curves!</details>

---## Phase 0 — Math Foundation Practice### 🎯 Your Turn: The Activation FunctionWrite the `sigmoid(x)` function, and its derivative `sigmoid_derivative(out)` where `out` is the pre-computed output of the sigmoid function (to save computation!).

In [ ]:
def sigmoid(x):    # YOUR CODE HERE    passdef sigmoid_derivative(out):    # Remember from the previous notebook: sigma' = sigma * (1 - sigma)    # YOUR CODE HERE    pass

---## Phase 1 — Algorithm Construction### Step 1: Network InitializationWe will build a 2-Layer Network: - `input_size` = 2- `hidden_size` = 2- `output_size` = 1We need weight matrices: `W1` (connects input to hidden) and `W2` (connects hidden to output). We also need bias vectors `b1` and `b2`.Initialize them with random values between -0.5 and 0.5.

In [ ]:
class NeuralNetwork:    def __init__(self):        # W1: 2 inputs -> 2 hidden (2x2 matrix)        self.W1 = [[random.uniform(-0.5, 0.5) for _ in range(2)] for _ in range(2)]        self.b1 = [random.uniform(-0.5, 0.5) for _ in range(2)]                # W2: 2 hidden -> 1 output (2x1 matrix)        self.W2 = [[random.uniform(-0.5, 0.5)] for _ in range(2)]        self.b2 = [random.uniform(-0.5, 0.5)]

### Step 2: Forward PropagationPass a 1D list of inputs `X` (e.g. `[1.0, 0.0]`) through the network.Calculate `hidden_z`, then apply `sigmoid` to get `hidden_a`. Calculate `output_z`, then apply `sigmoid` to get `output_a`.Return all of them, as they are needed for backpropagation!

In [ ]:
# Add to NeuralNetwork class# def forward(self, X):#     # 1. Hidden Layer#     hidden_a = [0.0, 0.0]#     for j in range(2):#         z = self.W1[0][j] * X[0] + self.W1[1][j] * X[1] + self.b1[j]#         hidden_a[j] = sigmoid(z)        #     # 2. Output Layer#     output_a = [0.0]#     z = self.W2[0][0] * hidden_a[0] + self.W2[1][0] * hidden_a[1] + self.b2[0]#     output_a[0] = sigmoid(z)    #     return hidden_a, output_a

### Step 3: BackpropagationWe use Mean Squared Error.The error signal at the output is `(output_a - target) * sigmoid_derivative(output_a)`.The error signal at a hidden node `j` is `(error_signal_out * W2[j][0]) * sigmoid_derivative(hidden_a[j])`.Update the weights: $W = W - \text{lr} \cdot \text{signal} \cdot \text{input\_to\_that\_weight}$

In [ ]:
# Add to NeuralNetwork class# def backward(self, X, target, hidden_a, output_a, lr=0.5):#     # 1. Output Layer Error#     out_error = (output_a[0] - target) * sigmoid_derivative(output_a[0])    #     # 2. Hidden Layer Errors#     hid_error = [0.0, 0.0]#     for j in range(2):#         hid_error[j] = (out_error * self.W2[j][0]) * sigmoid_derivative(hidden_a[j])        #     # 3. Update W2 and b2#     for j in range(2):#         self.W2[j][0] -= lr * out_error * hidden_a[j]#     self.b2[0] -= lr * out_error    #     # 4. Update W1 and b1#     for i in range(2):#         for j in range(2):#             self.W1[i][j] -= lr * hid_error[j] * X[i]#     for j in range(2):#         self.b1[j] -= lr * hid_error[j]

### Step 4: Training LoopLoop through epochs, passing each datapoint through `forward` and `backward`.

In [ ]:
# Add to NeuralNetwork class# def train(self, data, epochs=10000, lr=0.5):#     for epoch in range(epochs):#         for X, target in data:#             hidden_a, output_a = self.forward(X)#             self.backward(X, target, hidden_a, output_a, lr)

---## Phase 2 — Experimental Verification### The XOR Problem RevisitedIn Algorithms 27, we saw that a single linear/logistic layer CANNOT solve the XOR problem. Let's prove our Neural Network can!

In [ ]:
# xor_data = [#     ([0, 0], 0),#     ([0, 1], 1),#     ([1, 0], 1),#     ([1, 1], 0)# ]# nn = NeuralNetwork()# nn.train(xor_data, epochs=10000, lr=0.5)# print("Predictions after training:")# for X, target in xor_data:#     _, out = nn.forward(X)#     print(f"X={X} -> Pred: {out[0]:.3f} (Target: {target})")

---📚 **Next:** Algorithms 37-39 (Minimax Game Trees)

## End-of-Notebook ReviewRetrieve the core idea one more time before leaving. This final recall makes the next review easier.

In [ ]:
final_recall = ''  # summarize the most important idea in your own wordsconfidence = 3  # set 1-5 after answeringok = len(final_recall.strip()) >= 20store.record_attempt('lesson.algorithms_35_36_neural_networks', ok, confidence=confidence, item_type='end_review')mastery_dashboard(store=store, skills=SKILLS, next_notebook='37-39_minimax_game_trees.ipynb')